In [ ]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv

In [6]:
load_dotenv()

True

In [10]:
model = ChatGoogleGenerativeAI(model="gemini-flash-latest")

In [11]:
class BlogState(TypedDict):
    topic: str
    outline: str
    blog: str

In [12]:
def generate_outline(state: BlogState)-> BlogState:
    topic = state['topic']
    prompt = f"Generate a detailed outline for a blog post on the topic: {topic}"
    outline = model.invoke(prompt).content
    state['outline'] = outline
    return state

In [13]:
def generate_blog(state: BlogState)-> BlogState:
    outline = state['outline']
    prompt = f"Generate a blog post based on the following outline: {outline}"
    blog = model.invoke(prompt).content
    state['blog'] = blog
    return state

In [17]:
graph = StateGraph(BlogState)

#create nodes
graph.add_node("generate_outline", generate_outline)
graph.add_node("generate_blog", generate_blog)

#create edges
graph.add_edge(START, "generate_outline")
graph.add_edge("generate_outline", "generate_blog")
graph.add_edge("generate_blog", END)
workflow = graph.compile()


In [ ]:
initial_state = BlogState(topic="Write a blog post about the benefits of meditation in 50 words", outline="", blog="")
final_state = workflow.invoke(initial_state)
print(final_state)